In [3]:
import time
import pandas as pd
import numpy as np

DTYPES = {c: "float32" for c in [f"t{i}" for i in range(1,14)] + ["x1","x2","x3"]}
DTYPES.update({"deviceId": "str", "deviceType": "int8"})
DATA_FOLDER = '/net/tscratch/people/tutorial237/EnsembleAi2026/task3/data'
DATA_PATH    = f"{DATA_FOLDER}/data.csv"
DEVICES_PATH = f"{DATA_FOLDER}/devices.csv"

t0 = time.time()
chunk = next(pd.read_csv(DATA_PATH, dtype=DTYPES, chunksize=500_000).__iter__())
chunk["year"]  = chunk["timedate"].str[:4].astype("int16")
chunk["month"] = chunk["timedate"].str[5:7].astype("int8")

# Test the new aggregations
agg = chunk.groupby(["deviceId","year","month"]).agg(
    x2_mean      = ("x2", "mean"),
    x2_max       = ("x2", "max"),
    x2_idle_frac = ("x2", lambda s: (s == 0).mean()),
    x2_high_frac = ("x2", lambda s: (s > 0.5).mean()),
).reset_index()

print(f"One chunk: {time.time()-t0:.1f}s")
print(f"Estimated total: {(time.time()-t0) * 126 / 60:.1f} min")

One chunk: 1.1s
Estimated total: 2.3 min


In [6]:
"""
Enhanced aggregation — adds richer per-device monthly features
on top of the existing means/stds.

New features:
  x2_max        — peak load reading in the month
  x2_median     — median load (robust to spikes)
  x2_idle_frac  — fraction of readings where x2 == 0 (device off)
  x2_high_frac  — fraction of readings where x2 > 0.5 (high load)
  x2_q75        — 75th percentile load
  x1_max        — peak operating frequency

Output: monthly_agg_v2.parquet
"""

import gc
import time
import numpy as np
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_FOLDER = '/net/tscratch/people/tutorial237/EnsembleAi2026/task3/data'
DATA_PATH    = f"{DATA_FOLDER}/data.csv"
DEVICES_PATH = f"{DATA_FOLDER}/devices.csv"
CHUNK_SIZE   = 500_000
OUT_PATH     = "monthly_agg_v2.parquet"
# ───────────────────────────────────────────────────────────────────────────────

TEMP_COLS = [f"t{i}" for i in range(1, 14)]
DTYPES    = {c: "float32" for c in TEMP_COLS + ["x1", "x2", "x3"]}
DTYPES.update({"deviceId": "str", "deviceType": "int8"})


def aggregate_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk["year"]  = chunk["timedate"].str[:4].astype("int16")
    chunk["month"] = chunk["timedate"].str[5:7].astype("int8")

    return chunk.groupby(["deviceId", "year", "month"]).agg(
        # ── target ────────────────────────────────────────────────────────────
        x2_mean      = ("x2", "mean"),
        x2_std       = ("x2", "std"),
        x2_count     = ("x2", "count"),
        x2_max       = ("x2", "max"),
        x2_median    = ("x2", "median"),
        x2_q75       = ("x2", lambda s: s.quantile(0.75)),
        x2_idle_frac = ("x2", lambda s: (s == 0).mean()),
        x2_high_frac = ("x2", lambda s: (s > 0.5).mean()),

        # ── temperature sensors ───────────────────────────────────────────────
        t1_mean  = ("t1",  "mean"),  t1_std  = ("t1",  "std"),
        t2_mean  = ("t2",  "mean"),  t2_std  = ("t2",  "std"),
        t3_mean  = ("t3",  "mean"),  t3_std  = ("t3",  "std"),
        t4_mean  = ("t4",  "mean"),  t4_std  = ("t4",  "std"),
        t5_mean  = ("t5",  "mean"),  t5_std  = ("t5",  "std"),
        t6_mean  = ("t6",  "mean"),  t6_std  = ("t6",  "std"),
        t7_mean  = ("t7",  "mean"),  t7_std  = ("t7",  "std"),
        t8_mean  = ("t8",  "mean"),  t8_std  = ("t8",  "std"),
        t9_mean  = ("t9",  "mean"),  t9_std  = ("t9",  "std"),
        t10_mean = ("t10", "mean"),  t10_std = ("t10", "std"),
        t11_mean = ("t11", "mean"),  t11_std = ("t11", "std"),
        t12_mean = ("t12", "mean"),  t12_std = ("t12", "std"),
        t13_mean = ("t13", "mean"),  t13_std = ("t13", "std"),

        # ── operating features ────────────────────────────────────────────────
        x1_mean    = ("x1", "mean"),
        x1_std     = ("x1", "std"),
        x1_max     = ("x1", "max"),
        x3_mode    = ("x3", lambda s: s.mode().iloc[0] if len(s) else np.nan),
        deviceType = ("deviceType", "first"),
    ).reset_index()


def combine_partial_aggs(chunks: list) -> pd.DataFrame:
    raw = pd.concat(chunks, ignore_index=True)
    return raw.groupby(["deviceId", "year", "month"]).agg(
        x2_mean      = ("x2_mean",      "mean"),
        x2_std       = ("x2_std",       "mean"),
        x2_count     = ("x2_count",     "sum"),
        x2_max       = ("x2_max",       "max"),    # true max across chunks
        x2_median    = ("x2_median",    "mean"),   # approx
        x2_q75       = ("x2_q75",       "mean"),   # approx
        x2_idle_frac = ("x2_idle_frac", "mean"),
        x2_high_frac = ("x2_high_frac", "mean"),

        t1_mean  = ("t1_mean",  "mean"),  t1_std  = ("t1_std",  "mean"),
        t2_mean  = ("t2_mean",  "mean"),  t2_std  = ("t2_std",  "mean"),
        t3_mean  = ("t3_mean",  "mean"),  t3_std  = ("t3_std",  "mean"),
        t4_mean  = ("t4_mean",  "mean"),  t4_std  = ("t4_std",  "mean"),
        t5_mean  = ("t5_mean",  "mean"),  t5_std  = ("t5_std",  "mean"),
        t6_mean  = ("t6_mean",  "mean"),  t6_std  = ("t6_std",  "mean"),
        t7_mean  = ("t7_mean",  "mean"),  t7_std  = ("t7_std",  "mean"),
        t8_mean  = ("t8_mean",  "mean"),  t8_std  = ("t8_std",  "mean"),
        t9_mean  = ("t9_mean",  "mean"),  t9_std  = ("t9_std",  "mean"),
        t10_mean = ("t10_mean", "mean"),  t10_std = ("t10_std", "mean"),
        t11_mean = ("t11_mean", "mean"),  t11_std = ("t11_std", "mean"),
        t12_mean = ("t12_mean", "mean"),  t12_std = ("t12_std", "mean"),
        t13_mean = ("t13_mean", "mean"),  t13_std = ("t13_std", "mean"),

        x1_mean    = ("x1_mean",    "mean"),
        x1_std     = ("x1_std",     "mean"),
        x1_max     = ("x1_max",     "max"),
        x3_mode    = ("x3_mode",    lambda s: s.mode().iloc[0] if len(s) else np.nan),
        deviceType = ("deviceType", "first"),
    ).reset_index()

In [7]:
t_start = time.time()
chunks  = []

for i, chunk in enumerate(
    pd.read_csv(DATA_PATH, dtype=DTYPES, chunksize=CHUNK_SIZE)
):
    t0  = time.time()
    agg = aggregate_chunk(chunk)
    chunks.append(agg)
    elapsed = time.time() - t0
    total   = time.time() - t_start
    print(f"  chunk {i+1:>3}: {len(chunk):>8,} rows → "
            f"{len(agg):>5,} groups | "
            f"chunk: {elapsed:.1f}s | elapsed: {total/60:.1f}min")
    del chunk
    gc.collect()

print("\nCombining partial aggregations ...")
monthly = combine_partial_aggs(chunks)

print("Merging device metadata ...")
devices = pd.read_csv(DEVICES_PATH)
monthly = monthly.merge(devices, on="deviceId", how="left")

monthly.to_parquet(OUT_PATH, index=False)

print(f"\nDone in {(time.time()-t_start)/60:.1f} min")
print(f"  Shape   : {monthly.shape}")
print(f"  Columns : {list(monthly.columns)}")
print(f"  Saved   : {OUT_PATH}")
print(f"\nSanity check:")
print(monthly.groupby(["year","month"])["deviceId"].count().to_string())

  chunk   1:  500,000 rows →    60 groups | chunk: 0.5s | elapsed: 0.0min
  chunk   2:  500,000 rows →    59 groups | chunk: 0.5s | elapsed: 0.0min
  chunk   3:  500,000 rows →    61 groups | chunk: 0.5s | elapsed: 0.1min
  chunk   4:  500,000 rows →    59 groups | chunk: 0.5s | elapsed: 0.1min
  chunk   5:  500,000 rows →    59 groups | chunk: 0.5s | elapsed: 0.1min
  chunk   6:  500,000 rows →    59 groups | chunk: 0.5s | elapsed: 0.1min
  chunk   7:  500,000 rows →    65 groups | chunk: 0.5s | elapsed: 0.2min
  chunk   8:  500,000 rows →    59 groups | chunk: 0.5s | elapsed: 0.2min
  chunk   9:  500,000 rows →    62 groups | chunk: 0.5s | elapsed: 0.2min
  chunk  10:  500,000 rows →    58 groups | chunk: 0.5s | elapsed: 0.2min
  chunk  11:  500,000 rows →    62 groups | chunk: 0.5s | elapsed: 0.2min
  chunk  12:  500,000 rows →    63 groups | chunk: 0.5s | elapsed: 0.3min
  chunk  13:  500,000 rows →    59 groups | chunk: 0.5s | elapsed: 0.3min
  chunk  14:  500,000 rows →    59 gro